# Extend the hourly PFC with an `RO` column

Takes `data/shaped_curve_wide_hourly_2026-08-26.csv` (hourly power/gas/EUA curves) and
appends ten columns describing the Retribución a la Operación in €/MWhE for the quarter
each hour falls in, seen from **as-of 2026-08-26**:

- **`RO`** — the reference (day-weighted) estimate, plus **`RO_fixed`** / **`RO_float`**, the
  two scenarios it blends, and **`RO_open_pct`**, how much of it can still move.
- **`RO_<commodity>_plus` / `_minus`** — the same RO recalculated against a PFC whose power,
  gas or EUA curve has been shifted by ±1 (section 4).

RO is a *quarterly* figure, so the column is constant within a quarter and steps at each
quarter boundary. Quarters already published in `data/ro_realised.xlsx` (26Q1–26Q3 at this
as-of) carry the **realised** BOE value; the rest are estimated from futures + this same PFC.

Writes `output/shaped_curve_wide_hourly_2026-08-26_with_RO.csv`. The input file is not modified.

In [ ]:
import os
import tempfile
from datetime import date
from pathlib import Path

import pandas as pd

# Read market data (futures.csv, the PFC, holidays.csv) from the repo's data/ rather than
# notebooks/data/: the package resolves it against the working directory.
os.environ["RO_DATA_DIR"] = str((Path("..") / "data").resolve())

from ro_calculation import load_price_frames, run_asof

ASOF = date(2026, 8, 26)

PFC_IN = Path("..") / "data" / "shaped_curve_wide_hourly_2026-08-26.csv"
OUT_DIR = Path("..") / "output"
PFC_OUT = OUT_DIR / f"{PFC_IN.stem}_with_RO.csv"

OUT_DIR.mkdir(exist_ok=True)

## 1. Load the PFC and align its column names

`gather_price_components` matches PFC columns against the commodity names `power` / `gas` /
`EUA` by **string equality**. This file names them `power_EEX` / `gas_EEX` / `EUA_ICE`, which
match nothing — `use_pfc=True` would silently fall back to futures-only and then raise for the
far-dated quarters that have no EUA quotes at all. So rename, and write the renamed copy out
for `run_asof` to read back (it takes a path, not a frame).

In [ ]:
pfc = pd.read_csv(PFC_IN, parse_dates=["datetime"])
print(f"{len(pfc):,} hours, {pfc['datetime'].min()} -> {pfc['datetime'].max()}")
print("columns:", pfc.columns.tolist())

PFC_RENAMES = {"power_EEX": "power", "gas_EEX": "gas", "EUA_ICE": "EUA"}
pfc_renamed = pfc.rename(columns=PFC_RENAMES)
pfc_renamed_path = OUT_DIR / f"{PFC_IN.stem}_renamed.csv"
pfc_renamed.to_csv(pfc_renamed_path, index=False)
pfc.head()

## 2. Load futures, and drop one bad EUA duplicate

`data/futures.csv` carries ICE's *rolling* front-month symbols. On a handful of roll-boundary
days two symbols resolve to the same contract with different closes — e.g. on 2026-07-27 both
`/ECF<4>` (82.96) and `/ECF<5>` (82.33) map to Dec-26. `_decompose_product` refuses to guess
between conflicting closes and raises, which blocks `q4_26` and `q1_27` (their EUA leg *is*
Dec-26).

Keeping the **higher-volume** row resolves it, and agrees with the roll arithmetic: standing in
August, Dec-26 is the front-*5* month, so `/ECF<5>` is the correct mapping and `/ECF<4>` is the
stale label. This is a workaround for a feed artifact — the real fix belongs upstream in
whatever produces `futures.csv`.

In [ ]:
df_power, df_gas, df_eua = load_price_frames()

PRODUCT_DAY = ["delivery_date", "end_delivery_date", "market_date"]
conflicts = df_eua.groupby(PRODUCT_DAY)["close"].transform("nunique") > 1
print(f"EUA rows with a conflicting same-day close: {int(conflicts.sum())}")

df_eua = df_eua.sort_values("volume").drop_duplicates(subset=PRODUCT_DAY, keep="last")

EUA rows with a conflicting same-day close: 15


## 3. One RO per quarter spanned by the PFC

`run_asof` checks `data/ro_realised.xlsx` first: a quarter that has already started as of
`ASOF` returns its published BOE value and is not calculated at all (`realised=True`, and its
three scenarios are identical since a published RO can no longer move). Everything else is
estimated, with this PFC supplying the still-open portion of each leg.

`RO_fixed` is the RO implied by the closes already observed, NaN until at least one full leg
has elapsed; `RO_float` is the RO if today's curve held for the whole of every window, NaN
once every window has closed. `RO` is the day-weighted blend of the two and is always
defined; `RO_open_pct` is the share of it still open, 0 for a realised quarter.

In [ ]:
def quarter_code(ts: pd.Timestamp) -> str:
    """Timestamp -> the target_quarter code it belongs to, e.g. 'q3_26'."""
    return f"q{(ts.month - 1) // 3 + 1}_{ts.year % 100:02d}"


def ro_open_pct(result: dict) -> float:
    """Share of the RO still open, in %. Zero for a realised quarter."""
    ro_row = result["df"].loc[result["df"]["commodity"] == "RO"].iloc[0]
    return 100.0 * float(ro_row["pct_open"])


pfc["quarter"] = pfc["datetime"].map(quarter_code)
quarters = list(dict.fromkeys(pfc["quarter"]))

results = {
    q: run_asof(ASOF, df_power, df_gas, df_eua, q, use_pfc=True, pfc_csv_path=pfc_renamed_path)
    for q in quarters
}

ro_by_quarter = pd.DataFrame(
    [
        {
            "quarter": q,
            "realised": r["realised"],
            "RO": r["ro_scenarios"]["reference"],
            "RO_open_pct": ro_open_pct(r),
            "RO_fixed": r["ro_scenarios"]["fixed"],
            "RO_float": r["ro_scenarios"]["float"],
            "hours": int((pfc["quarter"] == q).sum()),
        }
        for q, r in results.items()
    ]
)
ro_by_quarter


## 4. Sensitivities: a ±1 bump on each commodity curve

Six more RO figures per quarter, each the whole calculation re-run against a PFC whose `power`,
`gas` or `EUA` column has been shifted by `BUMP_SIZE = 1` — €/MWh for power and gas, €/t for EUA —
across every hour.

The bump lands on the **curve**, not on the final averaged price, so it moves only the still-open
portion of each leg; the already-observed futures closes that make up the fixed part are untouched.
A quarter at `RO_open_pct = 100` therefore shows the full sensitivity, `q4_26` at ~46% shows a
little under half of it, and a realised quarter shows none at all — `run_asof` returns its
published BOE value whatever the PFC says, so its six bump columns all equal `RO`.

Six fan-outs across every quarter, so this cell takes a couple of minutes.

In [ ]:
BUMP_SIZE = 1.0
BUMP_COMMODITIES = ("power", "gas", "EUA")

bump_columns = []
with tempfile.TemporaryDirectory() as tmp:
    for commodity in BUMP_COMMODITIES:
        for sign, tag in ((1, "plus"), (-1, "minus")):
            shifted = pfc_renamed.copy()
            shifted[commodity] = shifted[commodity] + sign * BUMP_SIZE
            shifted_path = Path(tmp) / f"{commodity}_{tag}.csv"
            shifted.to_csv(shifted_path, index=False)

            bumped = run_asof(
                ASOF, df_power, df_gas, df_eua, quarters,
                use_pfc=True, pfc_csv_path=shifted_path,
            )
            column = f"RO_{commodity.lower()}_{tag}"
            ro_by_quarter[column] = ro_by_quarter["quarter"].map(
                {q: r["ro_scenarios"]["reference"] for q, r in bumped.items()}
            )
            bump_columns.append(column)

ro_by_quarter


## 5. Broadcast onto the hourly grid and write out

In [ ]:
RO_COLUMNS = ["RO", "RO_open_pct", "RO_fixed", "RO_float"] + bump_columns

lookup = ro_by_quarter.set_index("quarter")
for column in RO_COLUMNS:
    pfc[column] = pfc["quarter"].map(lookup[column])

# RO_fixed / RO_float are legitimately NaN (no full leg elapsed yet / no window still open),
# so only the always-defined columns are checked for completeness.
assert pfc[["RO", "RO_open_pct"]].notna().all().all(), "every hour must carry an RO"
assert pfc.groupby("quarter")[RO_COLUMNS].nunique(dropna=False).eq(1).all().all(), \
    "every RO column must be constant within a quarter"

out = pfc.drop(columns="quarter")
out.to_csv(PFC_OUT, index=False)
print(f"Wrote {len(out):,} rows x {len(out.columns)} cols to {PFC_OUT}")
out.head()


## 6. Sanity check: the RO steps only at quarter boundaries

Every row where `RO` differs from the hour before should be the first hour of a quarter —
1 Jan / 1 Apr / 1 Jul / 1 Oct at 00:00.

In [ ]:
steps = out[out["RO"].ne(out["RO"].shift())]
assert steps["datetime"].dt.month.isin([1, 4, 7, 10]).all()
assert (steps["datetime"].dt.day == 1).all() and (steps["datetime"].dt.hour == 0).all()

steps[["datetime", "RO"]].reset_index(drop=True)

,datetime,RO
0,2026-01-01,52.963000
1,2026-04-01,89.486000
2,2026-07-01,55.377000
3,2026-10-01,56.734351
4,2027-01-01,71.698840
5,2027-04-01,81.193427
6,2027-07-01,61.254953
7,2027-10-01,60.201192
8,2028-01-01,56.619296
9,2028-04-01,64.227396
